# Data Analysis & Modeling: Choose Two Experiences for Each Request

**Original mock inspired by collected interview topics, not a reported Roblox coding question.** All data are synthetic. Format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. Explain your assumptions and choices while coding.

## Interviewer prompt

A game platform has logged recommendation requests. Each request contains several displayed experience cards, and each card has a label indicating whether the user joined that experience within ten minutes.

**Problem 1 — Data analysis (13 minutes).** Inspect the two exports, resolve duplicate records using the policy below, join experience metadata, and summarize join rates with their sample sizes.

**Problem 2 — Modeling and ranking (22 minutes).** Train a simple join-probability model and evaluate it on unseen users. Then choose up to two cards per held-out request and compare this diagnostic selection with keeping the earliest historical positions. Explain what this offline comparison can and cannot establish.

This is an original practice exercise, not a claim that Roblox asks this exact coding question. Work in ordinary pandas and scikit-learn; a neural network and a complete recommendation service are not required.

- Review: missing values, many-to-one joins, train-only pipelines, user-grouped splits and binary metrics.
- New situation: conflicting event IDs require quarantine, not just drop_duplicates().
- New operations: sort_values, groupby(...).head(2), and reindex for request-level evaluation.
- New reasoning: global classification metrics and within-request ranking metrics answer different questions.

| Coding time | Focus |
| --- | --- |
| 0–4 min | Understand the exports |
| 4–9 min | Clean events without inventing a version |
| 9–13 min | Join metadata and examine join rates |
| 13–17 min | Choose the holdout unit |
| 17–24 min | Fit a reproducible pointwise model |
| 24–28 min | Separate prediction quality from ranking input |
| 28–35 min | Select two cards and explain the handoff |

Suggested checkpoints only; they are not official grading criteria. The two problems are data analysis (Steps 1–3) and modeling (Steps 4–7).

**Practice version:** write your own imports, then complete each exercise and run its check cell. The questions are part of the mock; explain your answer before moving on. Run All intentionally stops at unfinished exercises.

## Data contract

The supplied files are **impressions.csv** and **experiences.csv**. One impression is one displayed experience card. One request contains several cards for one user; an experience can appear in many requests. A user must not appear in both training and test sets.

| File / field | Meaning |
| --- | --- |
| impressions.impression_id | Event identifier; exact re-exports and conflicting records are deliberately included |
| impressions.user_id / request_id / experience_id | User, recommendation batch and experience identifiers; not numeric predictors |
| impressions.prior_genre_minutes | Historical play time in this genre, available before the impression; negative means unknown |
| impressions.friends_playing | Friends playing at impression time |
| impressions.position | Historical display position, starting at 1 |
| impressions.device | Device category; may be missing |
| impressions.joined | Target: 1 if the user joined within ten minutes, otherwise 0; all windows have finished |
| impressions.minutes_after_impression | Future behavior; unavailable when scoring and excluded from features |
| experiences.experience_id | Unique metadata key |
| experiences.genre / experience_age_days | Category and age available at impression time |

**Conflict policy for this exercise:** remove exact duplicate rows first. If an impression_id still occurs more than once, quarantine every row for that ID because there is no timestamp or authoritative version. Report the number of rows and event IDs affected. Do not arbitrarily keep the first or last conflicting version. Do not edit the source CSVs.

After cleaning, requests can contain different numbers of cards. Never assume every request has four cards or that a particular row count must survive. Check that each request belongs to one user and contains no repeated experience.

For the ranking diagnostic, score all candidate cards as if their **position feature were 1**. This removes differing historical position values from the scoring input, but does not identify how users would respond to a different display order. Logged labels were collected under the old policy.


## Setup

Use the workspace root as the working directory and select **.venv-codesignal/bin/python**. Inputs are in **data/codesignal_roblox_recommendation/**. No downloads, data generation, GPU or new dependencies are needed.

Write the imports you need, define **SEED = 42**, and set **DATA_DIR** to the supplied directory. Include setup in your timed attempt. Restart the kernel first so earlier notebooks do not hide missing imports.

Do not edit the source CSVs. These exercises deliberately reuse existing files to focus on new decisions rather than a new dataset generator.


In [19]:
# Write your own imports, SEED and DATA_DIR.
from pathlib import Path
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split, GroupShuffleSplit


SEED = 42

DATA_DIR = Path("data/codesignal_roblox_recommendation")

#raise NotImplementedError("Complete setup before loading the CSVs.")


## Problem 1 · Step 1: Understand the exports (0–4 min)

What does one row represent, and what is the prediction target? Load **impressions** and **experiences**. Inspect their shapes, column types, missing values, unique users and requests, exact duplicates, and negative historical play time. Which columns would be unavailable when a card is shown?

**Say before coding:** "I will identify the event grain and prediction time before deciding what can be a feature."

**Possible interviewer questions**

1. Why is user_id.count() not the number of users?
2. Does a repeated impression_id always mean two identical rows?
3. Can minutes_after_impression help predict joined?


In [6]:
# Implement the task above.
impressions = pd.read_csv(DATA_DIR / "impressions.csv")
experiences = pd.read_csv(DATA_DIR / "experiences.csv")
impressions.info()
display(impressions.head(), experiences.head(), impressions.isna().sum())
print("users:", impressions["user_id"].nunique())
print("requests:", impressions["request_id"].nunique())
print("exact duplicate rows:", impressions.duplicated().sum())
print("negative history:", impressions["prior_genre_minutes"].lt(0).sum())
display(impressions["joined"].value_counts(normalize=True))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2421 entries, 0 to 2420
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   impression_id             2421 non-null   int64  
 1   user_id                   2421 non-null   int64  
 2   request_id                2421 non-null   int64  
 3   experience_id             2421 non-null   int64  
 4   device                    2371 non-null   object 
 5   position                  2421 non-null   int64  
 6   prior_genre_minutes       2421 non-null   float64
 7   friends_playing           2421 non-null   int64  
 8   joined                    2421 non-null   int64  
 9   minutes_after_impression  2421 non-null   float64
dtypes: float64(2), int64(7), object(1)
memory usage: 189.3+ KB


,impression_id,user_id,request_id,experience_id,device,position,prior_genre_minutes,friends_playing,joined,minutes_after_impression
0,0,0,0,51,desktop,1,27.028748,1,1,20.215565
1,1,0,0,37,desktop,2,47.208345,1,1,13.380891
2,2,0,0,38,desktop,3,26.547296,1,0,0.000000
3,3,0,0,27,desktop,4,17.268351,0,0,0.000000
4,4,1,1,41,mobile,1,23.604279,1,0,0.000000


,experience_id,genre,experience_age_days
0,0,adventure,1383
1,1,puzzle,1117
2,2,simulation,550
3,3,simulation,1451
4,4,simulation,616


impression_id                0
user_id                      0
request_id                   0
experience_id                0
device                      50
position                     0
prior_genre_minutes          0
friends_playing              0
joined                       0
minutes_after_impression     0
dtype: int64

users: 600
requests: 600
exact duplicate rows: 20
negative history: 60


joined
0    0.918629
1    0.081371
Name: proportion, dtype: float64

In [ ]:
assert experiences["experience_id"].is_unique
assert impressions["joined"].notna().all()
assert set(impressions["joined"]) == {0, 1}
assert impressions[["impression_id", "user_id", "request_id", "experience_id"]].notna().all().all()


## Problem 1 · Step 2: Clean events without inventing a version (4–9 min)

How would you distinguish repeated exports from conflicting events? Apply the contract's conflict policy to create **clean_impressions** and preserve **conflicts** for investigation. Report both kinds of removal, replace invalid historical durations with missing values, and explain why the surviving number of cards can differ between requests.

**Say before coding:** "I will remove identical copies, then quarantine all conflicting versions rather than guess which one is correct."

**Possible interviewer questions**

1. Why use duplicated(keep=False) after removing exact duplicates?
2. Why not keep the final CSV row for an event?
3. Does quarantine have a cost?


In [13]:
# Implement the task above.
clean_impressions = impressions.drop_duplicates().copy()

conflict_mask = clean_impressions['impression_id'].duplicated(keep=False)

conflicts = clean_impressions.loc[conflict_mask]

clean_impressions = clean_impressions.loc[~conflict_mask].copy()

clean_impressions["prior_genre_minutes"] = clean_impressions["prior_genre_minutes"].mask(
    clean_impressions["prior_genre_minutes"].lt(0)
)
display(clean_impressions)
display(clean_impressions.groupby("request_id").size().value_counts())

#raise NotImplementedError("Complete step 2, then run its checks.")


,impression_id,user_id,request_id,experience_id,device,position,prior_genre_minutes,friends_playing,joined,minutes_after_impression
0,0,0,0,51,desktop,1,27.028748,1,1,20.215565
1,1,0,0,37,desktop,2,47.208345,1,1,13.380891
2,2,0,0,38,desktop,3,26.547296,1,0,0.000000
3,3,0,0,27,desktop,4,17.268351,0,0,0.000000
4,4,1,1,41,mobile,1,23.604279,1,0,0.000000
...,...,...,...,...,...,...,...,...,...,...
2396,2395,598,598,45,mobile,4,14.966915,0,0,0.000000
2397,2396,599,599,45,mobile,1,72.967507,3,0,0.000000
2398,2397,599,599,15,mobile,2,51.930630,2,0,0.000000
2399,2398,599,599,2,mobile,3,1.011575,0,0,0.000000


4    599
3      1
Name: count, dtype: int64

In [ ]:
assert clean_impressions["impression_id"].is_unique
assert not clean_impressions["impression_id"].isin(conflicts["impression_id"]).any()
assert len(impressions) == exact_copies_removed + len(conflicts) + len(clean_impressions)
assert clean_impressions["prior_genre_minutes"].dropna().ge(0).all()


## Problem 1 · Step 3: Join metadata and examine join rates (9–13 min)

How would you attach one metadata record to each impression without multiplying events? Create **frame**, check the resulting grain, and report impression counts and join rates by genre and historical position. What can these summaries tell you, and what would be unsafe to conclude?

**Say before coding:** "I will keep the impression grain and place sample sizes next to rates so that small groups are not mistaken for strong evidence."

**Possible interviewer questions**

1. Why validate="many_to_one"?
2. Does a high position-1 join rate prove the first card is intrinsically best?
3. Why verify request_id maps to one user?


In [18]:
# Implement the task above.
frame = clean_impressions.merge(
    experiences, on='experience_id', how='left', validate='many_to_one'
)

display(frame.groupby("genre").agg(
    impressions=("impression_id", "size"), join_rate=("joined", "mean")
))

display(frame.groupby("position").agg(
    impressions=("impression_id", "size"), join_rate=("joined", "mean")
))



#raise NotImplementedError("Complete step 3, then run its checks.")


,impressions,join_rate
genre,,
adventure,605,0.089256
puzzle,946,0.087738
simulation,848,0.064858


,impressions,join_rate
position,,
1,599,0.113523
2,600,0.095000
3,600,0.058333
4,600,0.053333


In [ ]:
assert len(frame) == len(clean_impressions)
assert frame["impression_id"].is_unique
assert frame["experience_age_days"].notna().all()
assert frame["genre"].notna().all()


## Problem 2 · Step 4: Choose the holdout unit (13–17 min)

We want to estimate performance on users the model has never seen. Which features would you select, what split would support that goal, and how would you verify it? Create **numeric_features**, **categorical_features**, **X**, **y**, **train_idx**, **test_idx**, and the four train/test objects. Reserve about 20% of users for testing with a reproducible seed.

**Say before coding:** "I will split users, not individual impressions, and use positional split indices consistently for features and labels."

**Possible interviewer questions**

1. Why not split random rows?
2. Why use iloc with the indices returned by the splitter?
3. Does test_size=0.2 guarantee exactly 20% of rows?


In [20]:
# Implement the task above.
numeric_features = ["prior_genre_minutes", "friends_playing", "position", "experience_age_days"]
categorical_features = ["device", "genre"]

X = frame[numeric_features + categorical_features]
y = frame["joined"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(X,y, groups=frame["user_id"]))
X_train, X_test = X.iloc[train_idx], y.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]



#raise NotImplementedError("Complete step 4, then run its checks.")


In [ ]:
assert train_users.isdisjoint(test_users)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert set(frame.iloc[train_idx]["request_id"]).isdisjoint(frame.iloc[test_idx]["request_id"])
assert not {"impression_id", "user_id", "request_id", "experience_id", "joined", "minutes_after_impression"} & set(X.columns)
assert y_train.nunique() == 2 and y_test.nunique() == 2


## Problem 2 · Step 5: Fit a reproducible pointwise model (17–24 min)

How would you handle missing numeric values, missing categories and categories not seen during training? Build **model**, using a train-only preprocessing pipeline and a regularized LogisticRegression classifier. Fit **baseline**, a constant class-prior predictor. Explain what each model learns and why no preprocessing statistic should come from the test users.

**Say before coding:** "I will start with a simple probability model and a constant reference, keeping imputation and encoding inside the fitted pipeline."

**Possible interviewer questions**

1. Why median imputation rather than dropping every incomplete impression?
2. What does handle_unknown="ignore" do?
3. What does the prior baseline predict?


In [21]:
# Implement the task above.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=500, random_state=SEED)),
])
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)


raise NotImplementedError("Complete step 5, then run its checks.")


NameError: name 'Pipeline' is not defined

In [ ]:
np.testing.assert_array_equal(model.classes_, [0, 1])
learned = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned, X_train[numeric_features].median().to_numpy())
assert model.named_steps["classifier"].coef_.shape[0] == 1


## Problem 2 · Step 6: Separate prediction quality from ranking input (24–28 min)

Does the model improve on a constant predictor on held-out impressions? Produce **test_prob** and **scores** using ROC-AUC, average precision and log loss. Then create **ranking_scores** for the same rows with every position feature set to 1. Why should the original test probabilities and these hypothetical ranking scores be kept separate?

**Say before coding:** "I will evaluate observed-impression predictions first, then make the position convention for a hypothetical ranking explicit."

**Possible interviewer questions**

1. Why pass probabilities rather than thresholded 0/1 predictions to AUC and AP?
2. Why set position=1 for all ranking inputs?
3. Does better global AUC guarantee better top-two choices for each request?


In [ ]:
# Implement the task above.
raise NotImplementedError("Complete step 6, then run its checks.")


In [ ]:
assert len(test_prob) == len(X_test) == len(ranking_scores)
assert np.isfinite(test_prob).all() and np.isfinite(ranking_scores).all()
assert ((ranking_scores >= 0) & (ranking_scores <= 1)).all()
assert ranking_input["position"].eq(1).all()
pd.testing.assert_frame_equal(X_test, X.iloc[test_idx])


## Problem 2 · Step 7: Select two cards and explain the handoff (28–35 min)

For each held-out request, which up to two experiences would you select? Create **heldout**, **ranked**, **top_two**, and **request_scores**. Sort model scores from high to low and break ties by experience_id. Compare with selecting the first two historical positions, also breaking ties deterministically.

Report mean precision over the actual selected count (one card means a denominator of one), mean recall among requests with at least one observed join, and hit rate both for all requests and for positive requests only. Report how many zero-positive requests are excluded from recall. Keep requests with fewer than two surviving cards.

Finally, show a small request with its original positions, observed labels, scores and selected cards. If asked to ship this ranking, what evidence would you need beyond this notebook?

**Say before coding:** "I will evaluate within each request, name each denominator, and treat the result as a logged-data diagnostic rather than evidence that reordering causes more joins."

**Possible interviewer questions**

1. Why divide precision by the actual number selected?
2. How should recall handle a request with no observed positive?
3. Can this comparison prove the proposed ranking increases joins?


In [ ]:
# Implement the task above.
raise NotImplementedError("Complete step 7, then run its checks.")


In [ ]:
expected_sizes = heldout.groupby("request_id").size().clip(upper=2)
actual_sizes = top_two.groupby("request_id").size().reindex(expected_sizes.index, fill_value=0)
pd.testing.assert_series_equal(actual_sizes, expected_sizes)
assert not top_two.duplicated(["request_id", "experience_id"]).any()
assert request_scores["request_count"].eq(heldout["request_id"].nunique()).all()
assert set(top_two.index).issubset(heldout.index)


## Optional follow-ups — outside the 35-minute core

A new request arrives with three candidate cards. How would you reuse the fitted preprocessing and model without fitting again? Rank the supplied cards, including an unknown device category and one missing historical value. Do not claim that these scores are validated probabilities for an unseen category or a new display policy.


## Scope and sources

The user-supplied **Data Analysis & Modeling Coding Interview** guide supplies the format: data loading, analysis, feature engineering, a baseline model, evaluation and live explanation in a CodeSignal notebook. It does not specify these exact tasks or APIs.

The local interview collection separates reported **Question/Asked** topics from our explanatory **Answer** material. This mock adapts the topics for practice; it is not evidence that Roblox used this coding problem. Synthetic rates and thresholds are not real product statistics or policy.

This original mock combines themes from the collected Roblox interview notes: [report 1161498](https://www.1point3acres.com/bbs/thread-1161498-1-1.html) discusses training examples, features and evaluation in an ML-design context; [report 1128678](https://www.1point3acres.com/bbs/thread-1128678-1-1.html) includes interaction-data processing. These reports do **not** establish that this exact notebook, conflict policy or request-ranking coding exercise was asked. The exercise follows the user's supplied Data Analysis & Modeling interview format.
